## Running open weight model on reasoning prompts

### Model : DeepSeek R1 Distill Qwen 1.5B

### Installing dependencies

In [ ]:
!pip install -q -U transformers accelerate

### Imports

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

### GPU is required to load model fast

#### colab T4 gpu is enough

In [ ]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Please enable GPU in Colab.")

### Loading model

In [ ]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.float16,
    device_map = "auto"
)

### Inference function

In [ ]:
def ask(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = full_output[len(formatted_prompt):].strip()

    print(answer)

### Reasoning prompts

In [ ]:
prompt1 = "All roses are flowers. Some flowers fade quickly. Do all roses fade quickly?"
prompt2 = "A doctor gives you 3 pills and says take one every half hour. How long before all pills are taken?"
prompt3 = "Every time Rohan carries an umbrella, it rains. Should Rohan stop carrying an umbrella to prevent rain?"
prompt4 = "Aisha is older than Ben. Ben is older than Carlos. Carlos is older than Diana. Who should get the senior discount if it's given to the second youngest?"
prompt5 = "A disease affects 1 in 1000 people. A test for it is 99% accurate. You test positive. What is the probability you actually have the disease?"

### Prompt 1

#### Q. All roses are flowers. Some flowers fade quickly. Do all roses fade quickly?

#### A. No

In [ ]:
ask(prompt1)

### Prompt2

#### Q. A doctor gives you 3 pills and says take one every half hour. How long before all pills are taken?

#### A. 1 hour

In [ ]:
ask(prompt2)

### Prompt 3

#### Q. Every time Rohan carries an umbrella, it rains. Should Rohan stop carrying an umbrella to prevent rain?

#### A. No

In [ ]:
ask(prompt3)

### Prompt 4

#### Q. Aisha is older than Ben. Ben is older than Carlos. Carlos is older than Diana. Who should get the senior discount if it's given to the second youngest?

#### A. Carlos

In [ ]:
ask(prompt4)

### Prompt 5

#### Q. A disease affects 1 in 1000 people. A test for it is 99% accurate. You test positive. What is the probability you actually have the disease?

#### A. ~9%

In [ ]:
ask(prompt5)